# Exploracion: deteccion de URLs spam

Este notebook documenta el flujo del caso practico: carga del dataset, preprocesamiento de URLs, division train/test, SVM base, optimizacion y guardado del modelo.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from app import MODEL_PATH, METRICS_PATH, train_and_optimize
from utils import load_url_spam_dataset, tokenize_url

## 1. Carga del conjunto de datos

In [2]:
df = load_url_spam_dataset(PROJECT_ROOT / "data" / "raw" / "url_spam.csv")
print(df.shape)
df.head()

(2369, 2)


,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


In [3]:
df["is_spam"].value_counts(normalize=True).rename("proportion").to_frame().join(
    df["is_spam"].value_counts().rename("count")
)

,proportion,count
is_spam,,
False,0.897003,2125
True,0.102997,244


## 2. Preprocesamiento de enlaces

La funcion `tokenize_url` separa la URL por componentes y signos de puntuacion, elimina stopwords y aplica una lematizacion ligera sin depender de corpus externos.

In [4]:
sample_url = df.loc[0, "url"]
print(sample_url)
print(tokenize_url(sample_url))

https://briefingday.us8.list-manage.com/unsubscribe
['briefingday', 'us8', 'list', 'manage', 'unsubscribe']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["url"],
    df["is_spam"],
    test_size=0.2,
    random_state=42,
    stratify=df["is_spam"],
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

(1895,) (474,)
is_spam
False    0.897098
True     0.102902
Name: proportion, dtype: float64


## 3. SVM base

Primero se evalua un SVM con clasificador `SVC()` en parametros por defecto. El script final usa el mismo punto de partida, complementado con variables de URL.

In [6]:
baseline_text_model = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=tokenize_url, token_pattern=None, min_df=2, ngram_range=(1, 2))),
    ("classifier", SVC()),
])
baseline_text_model.fit(X_train, y_train)
print("Baseline text-only accuracy:", baseline_text_model.score(X_test, y_test))

Baseline text-only accuracy: 0.9451476793248945


## 4. Optimizacion y guardado

El entrenamiento final vive en `src/app.py`. Ejecuta un `GridSearchCV`, guarda train/test procesados, el modelo y las metricas.

In [7]:
results = train_and_optimize()
results["optimized_svm"]["best_params"]

Fitting 5 folds for each of 192 candidates, totalling 960 fits


{'classifier__C': 3,
 'classifier__class_weight': None,
 'classifier__gamma': 'scale',
 'classifier__kernel': 'linear',
 'features__tfidf__min_df': 1,
 'features__tfidf__ngram_range': (1, 1)}

In [8]:
metrics = pd.read_json(METRICS_PATH)
print(MODEL_PATH.exists(), METRICS_PATH.exists())
print("Modelo:", MODEL_PATH.relative_to(PROJECT_ROOT))
print("Metricas:", METRICS_PATH.relative_to(PROJECT_ROOT))
print("F1 spam optimizado:", results["optimized_svm"]["f1_spam"])

True True
Modelo: models\url_spam_svm_pipeline.joblib
Metricas: models\url_spam_svm_metrics.json
F1 spam optimizado: 0.7526881720430108


## 5. Prueba rapida de inferencia

In [9]:
model = joblib.load(MODEL_PATH)
examples = pd.Series([
    "https://www.reuters.com/investigates/special-report/health-coronavirus-britain-pub/",
    "https://briefingday.us8.list-manage.com/unsubscribe",
])
pd.DataFrame({"url": examples, "is_spam_prediction": model.predict(examples)})

,url,is_spam_prediction
0,https://www.reuters.com/investigates/special-r...,False
1,https://briefingday.us8.list-manage.com/unsubs...,True
